In [37]:
from unstructured.partition.pdf import partition_pdf
from IPython.display import Image, display
import base64
import pandas as pd
import io
import tabulate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
from collections import defaultdict
import json
import psycopg2
from pgvector.psycopg2 import register_vector
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from ragas import RunConfig
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder





C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_23028\3779772822.py:23: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_23028\3779772822.py:23: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_23028\3779772822.py:23: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. E

In [2]:
load_dotenv()

True

In [3]:
# gemini embedding 2 preview is layout aware and can recognise table markdown as a relational grid. 
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [4]:
llm = ChatOllama(model="mistral-small:24b")

In [5]:
llm1 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [6]:
llm2 = ChatOllama(model="llama3.1:latest")

In [7]:
llm3 = ChatOllama(
    model="qwen2.5:14b",
    temperature=0
)

In [43]:
chunks = partition_pdf(
    filename="pdfs/government-data-security-policies.pdf",
    strategy="hi_res", # table detection and extraction as text_as_html

    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=True, # extracts images and tables as base64 encoded strings

    chunking_strategy="by_title",
    max_characters=2000,
    combine_text_under_n_chars=1000,
    new_after_n_chars=1500
)
'''chunk by title chunks by section headings and titles. For structured documents, each document
element is already semantically sound. computaionally cheaper than semantic chunking which is more 
suitable for highly unstructured data. 
'''

'chunk by title chunks by section headings and titles. For structured documents, each document\nelement is already semantically sound. computaionally cheaper than semantic chunking which is more \nsuitable for highly unstructured data. \n'

In [44]:
chunks[0].metadata.orig_elements

In [ ]:
string = chunks[20].metadata.orig_elements[0].metadata.image_base64
display(Image(data=base64.b64decode(string)))

In [9]:
'''loops through each element in chunk. If element is table, converts text_as_html to markdown
which is generally better (lower token cost, more readable to llms)
'''
def modify_text(chunk):
    text = []
    for element in chunk.metadata.orig_elements:
        if element.category == "Table" and getattr(element.metadata, "text_as_html", None):
            df = pd.read_html(io.StringIO(element.metadata.text_as_html))[0]
            markdown = df.to_markdown(index=False)
            text.append(markdown)
        else:
            text.append(element.text)
    chunk.text = "\n\n".join(text)

In [10]:
def extract_coords(chunk):
    highlights = []
    orig_elements = chunk.metadata.orig_elements
    for element in orig_elements:
        page = element.metadata.page_number
        coords = element.metadata.coordinates.points

        width = element.metadata.coordinates.system.width
        height = element.metadata.coordinates.system.height

        xs = [p[0] for p in coords]
        ys = [p[1] for p in coords]
    
        clean_box = [
            round(max(0, float(min(xs))),2),
            round(max(0, float(min(ys))),2),
            round(min(width, float(max(xs))), 2),
            round(min(height, float(max(ys))), 2)
        ]
        highlights.append({
            "page": int(page),
            "bbox": clean_box,
            "width": int(width),
            "height": int(height)
        })
    setattr(chunk.metadata, "highlights", highlights)

In [11]:
def embed_text(chunk):
    vector = embeddings.embed_query(chunk.text)
    setattr(chunk.metadata, "vector", vector)

In [12]:
def number_chunk(chunk, number):
    setattr(chunk.metadata, "chunk_number", number)


In [13]:
def process_chunk(chunk, number):
    modify_text(chunk)
    embed_text(chunk)
    extract_coords(chunk)
    number_chunk(chunk, number)

In [14]:
for index, chunk in enumerate(chunks):
    process_chunk(chunk, index + 1)

In [18]:
import fitz
def visualize_chunk_highlights(pdf_path, chunk, output_path="preview.pdf"):
    doc = fitz.open(pdf_path)
    highlights = getattr(chunk.metadata, "highlights", [])

    for h in highlights:
        page_index = h['page'] - 1
        page = doc[page_index]
        
        # 1. Get the page size as PyMuPDF sees it (72 DPI)
        actual_w = page.rect.width
        actual_h = page.rect.height
        
        # 2. Get the original system size from your metadata
        orig_w = h['width']
        orig_h = h['height']
        
        # 3. SCALE the coordinates
        # Math: (Raw_Value / Original_System_Size) * Actual_PDF_Size
        raw_bbox = h['bbox'] # [x1, y1, x2, y2]
        
        scaled_bbox = [
            (raw_bbox[0] / orig_w) * actual_w, # x_min
            (raw_bbox[1] / orig_h) * actual_h, # y_min
            (raw_bbox[2] / orig_w) * actual_w, # x_max
            (raw_bbox[3] / orig_h) * actual_h  # y_max
        ]
        
        # 4. Draw the correctly scaled rectangle
        rect = fitz.Rect(scaled_bbox)
        page.draw_rect(rect, color=(1, 1, 0), fill=(1, 1, 0), overlay=True, width=0)

    doc.save(output_path)
visualize_chunk_highlights("government-data-security-policies.pdf", chunks[1], output_path="preview.pdf")

In [19]:
print(chunks[20].text)

| 0                       | 1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|:------------------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
chunks[20].metadata.highlights

[{'page': 22,
  'bbox': [234.95, 292.45, 2614.08, 3050.35],
  'width': 2894,
  'height': 4093}]

In [21]:
chunks[20].metadata.chunk_number

21

docker exec -it pgvector-db psql -U postgres

In [15]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [15]:
chunk = chunks[0]
chunk_embedding = chunk.metadata.vector
chunk_text = chunk.text
if chunk.metadata.image_base64:
    image_base64 = base64.b64decode(chunk.metadata.image_base64)
else: 
    image_base64 = None
highlights = chunk.metadata.highlights

In [16]:
try:
    cur.execute('''
        INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights)
        VALUES (%s, %s, %s, %s)
    ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights)))
    conn.commit()
except Exception as e: 
    print(f"Postgres Error: {e}")
    conn.rollback()

In [21]:
def store_chunk(chunk):
    chunk_embedding = chunk.metadata.vector
    chunk_text = chunk.text
    if chunk.metadata.image_base64:
        image_base64 = base64.b64decode(chunk.metadata.image_base64)
    else: 
        image_base64 = None
    highlights = chunk.metadata.highlights
    chunk_number = chunk.metadata.chunk_number
    try:
        cur.execute('''
            INSERT INTO chunk_store (chunk_embedding, chunk_text, image_base64, highlights, chunk_number)
            VALUES (%s, %s, %s, %s, %s)
        ''', (chunk_embedding, chunk_text, image_base64, json.dumps(highlights), chunk_number))
        conn.commit()
    except: 
        print("Error")
        conn.rollback()

In [23]:
for chunk in chunks:
    store_chunk(chunk)

In [26]:
cur.execute("SELECT chunk_text FROM chunk_store WHERE chunk_number = 1")
data = cur.fetchall()
data

[('GOVERNMENT    DATA SECURITY   POLICIES  This document contains general information for the   public only. It is not intended to be relied upon as a   comprehensive or deﬁnitive guide on each agency’s   policies and practices. The data security measures   implemented by each agency will differ depending on   various factors such as the sensitivity of the data and   the agency’s assessment of data security risks. The   Government may update the policies set out in this   document without publishing such updates to the   public.      Government Data Security Policies  |   1 \n\nGOVERNMENT DATA SECURITY POLICIES\n\nThis document contains general information for the public only. It is not intended to be relied upon as a comprehensive or deﬁnitive guide on each agency’s policies and practices. The data security measures implemented by each agency will differ depending on various factors such as the sensitivity of the data and the agency’s assessment of data security risks. The Government 

In [18]:
query = "data security risk management"

In [16]:
def search_chunks(query):
    query_vector = embeddings.embed_query(query)
    search_query = '''
        SELECT chunk_number, chunk_text, 1 - (chunk_embedding <=> %s) AS similarity
        FROM chunk_store
        ORDER BY chunk_embedding <=> %s
        LIMIT 5;
    '''
    try:
        cur.execute(search_query, (str(query_vector), str(query_vector)))
        data = cur.fetchall()
        df = pd.DataFrame(data, columns=["chunk_number", "chunk_text", "similarity"])
        return df
    except Exception as e:
        print(f"An error occurred: {e}")
        conn.rollback()
    

In [19]:
df = search_chunks(query)

In [20]:
df.head()


,chunk_number,chunk_text,similarity
0,4,Section 2: Technical and process measures to p...,0.712063
1,2,The Government takes its responsibility as a c...,0.696333
2,9,Government Data Security Policies | 8\n\n\n\nS...,0.695529
3,3,Government Data Security Policies | 2\n\n\n\nS...,0.689794
4,17,\n\nTechnical measures and the complementary p...,0.675025


In [40]:
def filter(df, threshold=0.1):
    max_similarity = df.iloc[0]["similarity"]
    mask = df["similarity"] >= (max_similarity - threshold)

    final_df = df[mask]
    return final_df

In [21]:
def filter1(query, df, threshold=0.1):
    max_similarity = df.iloc[0]["similarity"]
    mask = df["similarity"] >= (max_similarity - threshold)
    df = df[mask]

    reranker_model = CrossEncoder('Qwen/Qwen3-Reranker-4B')
    reranker_model.tokenizer.pad_token = reranker_model.tokenizer.eos_token
    reranker_model.model.config.pad_token_id = reranker_model.model.config.eos_token_id
    pairs = [[query, chunk_text] for chunk_text in df["chunk_text"]]
    df["rerank_score"] = reranker_model.predict(pairs)
    df = df.sort_values(by="rerank_score", ascending=False)

    return df


In [22]:
filter1(query, df)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5021.55it/s]
Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-Reranker-4B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,chunk_number,chunk_text,similarity,rerank_score
2,9,Government Data Security Policies | 8\n\n\n\nS...,0.695529,0.894531
0,4,Section 2: Technical and process measures to p...,0.712063,0.812500
1,2,The Government takes its responsibility as a c...,0.696333,0.628906
4,17,\n\nTechnical measures and the complementary p...,0.675025,0.382812
3,3,Government Data Security Policies | 2\n\n\n\nS...,0.689794,0.125977


In [30]:
lis = df["chunk_text"].tolist()
lis

['Section 2: Technical and process measures to protect data and prevent data compromises\n\nAgencies should implement the appropriate technical and process measures to protect 05 data against data security threats and prevent compromises of the conﬁdentiality and integrity of the data.\n\nAgencies should adopt a risk-based approach when implementing the technical and process measures. Technical measures are only eﬀective when the complementary process measures are implemented too (See Annex A).\n\nThe data security technical and process measures are to be implemented on top of any cybersecurity measures in place.\n\nData should be protected according to the risk level determined through a data security risk assessment. Agencies should ensure that data is adequately protected while minimising the impact to operations, resources and costs.\n\n06 Agencies should adopt the following strategies when implementing the technical measures and process measures to safeguard data:\n\na) Reduce the

In [28]:
cur.close()
conn.close()

In [23]:
def generate_context(dataframe):
    context_text = "\n\n".join(
        f"--- Chunk {row['chunk_number']} ---\n{row['chunk_text']}"
        for _, row in dataframe.iterrows()
    )
    return context_text

In [24]:
context_text = generate_context(df)

In [25]:
def generate_message(query, context_text):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.
    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).
    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.
    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  
    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.
    6. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    HumanMessage(content=f"""
    Context: 
    {context_text}

    Question: {query}
    """),

    ]
    return messages

In [26]:
messages = generate_message(query, context_text)

In [35]:
print(messages)

[SystemMessage(content='\n    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.\n\n    RULES:\n    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.\n    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).\n    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.\n    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  \n    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.\n    6. FORMATTING: Use clear headings and bullet points for complex answers.                          \n    ', additional_kwargs={}, response_

In [27]:
response = llm.invoke(messages)

In [28]:
print(response.content)

### Data Security Risk Management

Data security risk management is a critical aspect of ensuring the protection of data within government agencies. Here are the key points related to data security risk management:

- **Risk Assessment**: Agencies should perform data security risk assessments for their datasets as part of the Government ICT Risk Management Methodology. This involves identifying data security risks, evaluating them, implementing measures to mitigate these risks, assessing the effectiveness of the implemented measures, and managing the risks within acceptable limits [Chunk 3].

- **Methodology**: The Data Security Risk Assessment Methodology, which is part of the Government ICT Risk Management Methodology, should be used to conduct these assessments. This methodology helps agencies understand the specific risks associated with their datasets and implement appropriate safeguards [Chunk 3].

- **Frequency of Assessments**:
    - Agencies should conduct a data security risk

In [29]:
ragas_llm = LangchainLLMWrapper(ChatOllama(
    model="llama3-groq-tool-use",
    format="json",
    temperature=0,
    num_ctx=8192
))
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_23028\4180710712.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOllama(
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_23028\4180710712.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [30]:
generator = TestsetGenerator(
    llm=ragas_llm,
    embedding_model=ragas_embeddings
)

In [31]:
loader = PyPDFLoader("government-data-security-policies.pdf")
documents = loader.load()

Ignoring wrong pointing object 8 0 (offset 0)


In [32]:
testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=10
)
test_df = testset.to_pandas()

Generating Samples: 100%|██████████| 12/12 [00:29<00:00,  2.47s/it]


In [33]:
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What ar the goverment datasecurity policys?,[Government Data Security Policies |  1\nGO...,This document contains general information for...,Data Security Manager,MISSPELLED,LONG,single_hop_specific_query_synthesizer
1,What is IM on ICT&SS Management?,[The Government takes its responsibility as a ...,The Government's data security policies have b...,Data Security Analyst,POOR_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,What is the purpose of the Government ICT Risk...,[Section 1: Data Security Risk Management\n01 ...,The Government ICT Risk Management Methodology...,Data Security Manager,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
3,What are some common data security threats?,[Section 2: Technical and process measures \nt...,Data should be protected according to the risk...,Data Security Analyst,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
4,What are some effective strategies for protect...,[<1-hop>\n\nData file integrity verification \...,Effective strategies for protecting sensitive ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


In [41]:
evaluation_data = []
for index, row in test_df.iterrows():
    question = row.user_input
    ground_truth = row.reference

    retrieved_df = search_chunks(question)
    filtered_df = filter(retrieved_df)
    context_text = filtered_df["chunk_text"].tolist()

    context = generate_context(retrieved_df)
    messages = generate_message(question, context)
    response = llm.invoke(messages)

    evaluation_data.append({
        "question": question,
        "answer": response.content,
        "contexts": context_text,
        "ground_truth": ground_truth
    })
eval_dataset = Dataset.from_list(evaluation_data)
    

    

    

In [57]:
df = eval_dataset.to_pandas()
df.head()

,question,answer,contexts,ground_truth
0,What are the government's data security policies?,The Government's data security policies are ou...,[GOVERNMENT DATA SECURITY POLICIES This ...,The Government Data Security Policies document...
1,What is the Public Sector Data Security Review...,The Public Sector Data Security Review Committ...,[The Government takes its responsibility as a ...,The Public Sector Data Security Review Committ...
2,What is the purpose of performing a data secur...,The purpose of performing a data security risk...,[Government Data Security Policies | 2\n\n\n\n...,Data security risk assessments are conducted t...
3,What cybersecurity measures should be implemen...,"To protect sensitive information, several cybe...",[b) Partially hide the full data. Even if the ...,Agencies should implement the appropriate tech...
4,How do agencies ensure that access to sensitiv...,Agencies can ensure that access to sensitive d...,[23 Agencies should set predeﬁned limits for:\...,Agencies ensure that access to sensitive data ...


In [58]:
config = RunConfig(
    max_workers=4, 
    timeout=180,
    max_retries=2
)
metrics = [
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision
]
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=llm3,
    embeddings=ragas_embeddings,
    batch_size=2
)

Evaluating: 100%|██████████| 48/48 [10:14<00:00, 12.81s/it]


In [59]:
print(results)

{'faithfulness': 0.9137, 'answer_relevancy': 0.9210, 'context_recall': 0.9583, 'context_precision': 0.9420}


In [118]:
results_df = results.to_pandas()

In [119]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall,context_precision
0,whats the diffrence between goverment datasecu...,[GOVERNMENT DATA SECURITY POLICIES This ...,The price is $500,The Government Data Security Policies document...,0.0,0.554779,1.000000,1.000000
1,What is the purpose of hashing-with-salt in da...,[b) Partially hide the full data. Even if the ...,The price is $500,Hashing-with-salt irreversibly changes a data ...,0.0,0.504838,1.000000,1.000000
2,What is the purpose of performing data securit...,[Government Data Security Policies | 2\n\n\n\n...,The price is $500,Data security risk assessments are performed t...,0.0,0.497669,1.000000,0.700000
3,What is the purpose of Annex A in data securit...,[\n\nTechnical measures and the complementary ...,The price is $500,Annex A is a section that outlines technical a...,0.0,0.503971,1.000000,0.833333
4,How do email data protection tools and digital...,[Agencies should ensure that the digital water...,The price is $500,"Email data protection tools, such as Data Loss...",0.0,0.467337,0.666667,1.000000
5,What technical safeguards are required for ema...,[\n\nTechnical measures and the complementary ...,The price is $500,Email data protection requires implementing DL...,0.0,0.507101,0.500000,0.700000
6,What features should a DLP tool have for monit...,[36 Agencies should implement DLP tools in the...,The price is $500,A DLP tool should be able to read the security...,0.0,0.527337,1.000000,1.000000
7,What measures can be taken to monitor for viol...,[Government Data Security Policies | 8\n\n\n\n...,The price is $500,To monitor for violations of data access contr...,1.0,0.531811,1.000000,0.477778
8,How do Agencies ensure that sensitive data can...,[17 Agencies should implement processes to ens...,The price is $500,Agencies should ensure that sensitive data can...,0.0,0.529280,1.000000,0.750000
9,What is a dataset and how does it relate to da...,[| Terms |...,The price is $500,A dataset refers to a collection of data which...,0.0,0.528632,1.000000,0.916667


In [120]:
results_df.iloc[7]["retrieved_contexts"]

['Government Data Security Policies | 8\n\n\n\nSection 4: Log and monitor data transactions to detect risky or suspicious activity\n\nAgencies should log and monitor data transactions to detect suspicious or risky activities 28 and take action to prevent any potential data compromise.\n\nAgencies should do this by:\n\na) Maintaining logs and records to pinpoint risky activities, detect suspicious activities, and to assist in response and remediation; and\n\nb) Deploying tools to detect suspicious activity and alert the relevant parties to respond to such activities.\n\nSection 4.1: Maintain logs and records to pinpoint high-risk activities and to assist in response and remediation\n\nMaintain data lineage for data to support data incident management\n\nAgencies should keep data lineage records for data. Data lineage records help to identify 29 any unauthorised access, usage or modiﬁcation of data. Data lineage records support Agencies in managing and responding to data incidents.\n\nTh